---
title: "Chapter 17: Natural Language Processing"
---

::: {.callout-lo}

## Learning outcomes {.unnumbered}

By the end of this chapter, you should be able to:

- Explain what natural language processing is and why representing language presents challenges for machine learning.
- Explain how word embeddings represent relationships among words and use them to explore similarity, analogies, and bias.
- Compare bag-of-words, word-embedding, and text-embedding representations.
- Use a pretrained embedding model and cosine similarity to retrieve relevant text.
- Explain the difference between sparse and dense retrieval.
- Explain at a high level how a language model generates text and identify important limitations of large language models.
- Describe the purpose and components of retrieval-augmented generation (RAG).
- Implement a basic RAG pipeline that chunks documents, retrieves relevant evidence, constructs a prompt, and generates an answer.
- Diagnose whether a poor RAG response arose during retrieval or generation.

:::

**Imports**

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

## What is natural language processing?

Before arriving at this page today, you may already have interacted with several systems that process language. Perhaps your phone completed a sentence, an email service filtered spam, a search engine interpreted a query, or a chatbot answered a question. These tools feel quite different, but they all face the same basic challenge: **how can a computer do something useful with human language?**

:::: {.columns}

::: {.column width="50%"}
![](img/gmail-spam-example.png){fig-alt="An email interface marking a message as spam"}

A spam filter classifies incoming messages.
:::

::: {.column width="50%"}
![](img/voice-assistant-ex.png){fig-alt="A voice assistant responding to spoken language"}

A voice assistant turns language into an action.
:::

::::

**Natural language processing (NLP)** is the area of artificial intelligence concerned with analyzing and generating human language. NLP applications include search, translation, sentiment analysis, document classification, summarization, speech recognition, information extraction, and conversational assistants.

![](img/WhatisNLP.png){fig-align="center" width="600" fig-alt="Examples of natural language processing tasks"}

At first, this might sound like a matter of programming enough vocabulary and grammar rules. Language, however, rarely follows a tidy collection of rules.

### Why is language difficult?

Consider the sentence in the following exchange. To whom does *he* refer?

![](img/referential_ambiguity.png){fig-align="center" width="750" fig-alt="A conversation illustrating that a pronoun can have an ambiguous referent"}

A human reader looks beyond the pronoun. We use the surrounding conversation, common sense, and knowledge of the world. Even then, we sometimes need to ask for clarification.

Ambiguity is not limited to pronouns. Consider these genuine-style [ambiguous newspaper headlines](http://www.fun-with-words.com/ambiguous_headlines.html):

> KICKING BABY CONSIDERED TO BE HEALTHY

> MILK DRINKERS ARE TURNING TO POWDER

Why are they funny? In the first headline, *kicking* can describe the baby or an action performed on the baby. In the second, *turning to* can mean *choosing* or *transforming into*. The intended meanings are ordinary; the unintended interpretations are surprising.

The words themselves have not changed. What changes is how we connect them.

### Context changes meaning

What would you put in each blank?

> I went to the bank to deposit some ________.

> I sat on the bank of the ________.

The nearby words make *bank* mean different things. Language also permits the reverse problem: two sentences can express similar ideas with few words in common. *The assignment deadline passed while I was ill* and *I was sick when my homework was due* are likely asking about the same policy. A useful NLP system needs to handle both exact wording and paraphrases.

This observation will matter throughout the chapter. Search based only on matching words may miss a relevant passage, while a model that focuses only on broad similarity may overlook an important exact term.

### From language to numbers

Machine learning models operate on numbers, not directly on words. Before a model can classify, retrieve, or generate text, the system must decide how to represent language numerically. That representation determines which similarities and differences the model can notice.

No representation captures every aspect of meaning. The right question is therefore not “Does the computer truly understand this sentence?” but “Does this representation retain the information needed for our task?” We begin with representations based on word counts and then move to embeddings designed to capture some aspects of meaning.

### A running example

Suppose we are building a question-answering assistant for a fictional university course. The assistant should answer questions using the course policies below. This small collection lets us inspect every stage of the system; the same ideas apply to much larger collections.

In [2]:
documents = pd.DataFrame(
    [
        {
            "doc_id": "late-work",
            "title": "Late work",
            "text": (
                "Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. "
                "If illness or an emergency prevents you from submitting, contact the course coordinator "
                "before the deadline when possible. Do not include private medical details in your message."
            ),
        },
        {
            "doc_id": "office-hours",
            "title": "Office hours",
            "text": (
                "Teaching assistants hold office hours on Mondays and Wednesdays. Office hours are for "
                "conceptual questions and debugging help. Bring a minimal example of your problem, but do "
                "not expect the teaching assistant to complete an assignment for you."
            ),
        },
        {
            "doc_id": "collaboration",
            "title": "Collaboration",
            "text": (
                "You may discuss assignment ideas with classmates, but the work you submit must be your own. "
                "Do not share code or written solutions. Acknowledge classmates or other sources that helped "
                "you understand an idea."
            ),
        },
        {
            "doc_id": "recordings",
            "title": "Lecture recordings",
            "text": (
                "Lecture recordings are posted on the course media page after class. Recordings are a "
                "supplement rather than a replacement for attending class. Technical failures may occasionally "
                "prevent a recording from being posted."
            ),
        },
        {
            "doc_id": "regrade",
            "title": "Regrade requests",
            "text": (
                "Submit a regrade request through the grading portal within seven days of receiving your mark. "
                "Explain the specific grading issue and refer to the rubric. A regrade may result in the mark "
                "increasing, decreasing, or remaining unchanged."
            ),
        },
    ]
)

documents[["doc_id", "title", "text"]]

,doc_id,title,text
0,late-work,Late work,Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency preven...
1,office-hours,Office hours,Teaching assistants hold office hours on Mondays and Wednesdays. Office hours are for conceptual questions and debug...
2,collaboration,Collaboration,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."
3,recordings,Lecture recordings,Lecture recordings are posted on the course media page after class. Recordings are a supplement rather than a replac...
4,regrade,Regrade requests,Submit a regrade request through the grading portal within seven days of receiving your mark. Explain the specific g...


## From word counts to embeddings

In Chapter 6, we used bag-of-words features to represent a document by the words it contains. That representation works well for many classification problems. A related representation, **term frequency–inverse document frequency (TF–IDF)**, gives less weight to terms that occur in many documents and more weight to terms that help distinguish a particular document.

Bag-of-words and TF–IDF are **sparse representations**: there is one feature for each term in the vocabulary, and most entries are zero. They capture lexical overlap well. If both a query and a document contain the unusual phrase *regrade request*, a sparse representation provides strong evidence that they are related.

However, sparse representations do not naturally recognize that *I was sick when the assignment was due* is related to a policy containing *illness*, *deadline*, and *submitting*. The meanings are related even though the exact words differ.

### Learning relationships among words

A bag-of-words vocabulary gives *happy* and *joyful* separate coordinates. Nothing in those coordinates tells a model that the words are related. **Word embeddings** take a different approach: they represent each word with a short, dense vector and try to place words used in similar contexts near one another.

This idea is often summarized by the **distributional hypothesis**:

> You shall know a word by the company it keeps. — J. R. Firth, 1957

Suppose you did not know the word *glorp*, but repeatedly saw sentences such as *The movie was glorp and moving*, *What a delightful, glorp story*, and *We loved the glorp ending*. The surrounding words provide clues about how *glorp* is being used. Word-embedding algorithms learn from this kind of co-occurrence at a much larger scale.

**word2vec** is a family of algorithms for learning word embeddings by predicting words from their surrounding context, or surrounding words from a target word. We will not study its training algorithm here. Instead, we will explore what a pretrained word2vec model learned.

### Exploring pretrained word2vec embeddings

The following examples use vectors from the historical `word2vec-google-news-300` model, which was trained on roughly 100 billion words from Google News. The complete model contains about three million words and phrases and requires more than 1.5 GB of storage. To keep the chapter reproducible, this repository contains a 410-word subset chosen for the demonstrations below. It contains the original 300-dimensional vectors, but it is not a general-purpose vocabulary.

Phrases in this vocabulary use underscores, as in `computer_programmer`. Because these embeddings were learned from news text rather than written by hand, their geometry reflects statistical patterns in that corpus.

In [3]:
from gensim.models import KeyedVectors

word_vectors = KeyedVectors.load("data/google-news-word2vec-subset.kv")
print(f"Vocabulary in course subset: {len(word_vectors):,} words and phrases")
print(f"Dimensions per word: {word_vectors.vector_size}")
word_vectors["UBC"][:10]

Vocabulary in course subset: 410 words and phrases
Dimensions per word: 300


array([-0.3828125 , -0.18066406,  0.10644531,  0.4296875 ,  0.21582031,
       -0.10693359,  0.13476562, -0.08740234, -0.14648438, -0.09619141],
      dtype=float32)

The vector for `UBC` contains 300 numbers. Unlike a bag-of-words vector, it is short and mostly nonzero. A single coordinate is not meant to have a label such as *university-ness*. Relationships emerge from the vector as a whole.

One way to explore those relationships is to ask for the vectors with the largest cosine similarity.

In [4]:
word_vectors.most_similar("UBC", topn=5)

[('UVic', 0.788647472858429),
 ('SFU', 0.7588528394699097),
 ('Simon_Fraser', 0.7356574535369873),
 ('UFV', 0.6880435943603516),
 ('VIU', 0.6778583526611328)]

The nearby vectors represent universities and related institutions rather than dictionary synonyms for `UBC`. They are close because they occurred in similar news contexts. The nearest neighbours would change if we trained word2vec on medical notes, novels, or social-media posts.

We can also compare selected pairs directly.

In [5]:
word_pairs = [("Canada", "hockey"), ("Japan", "hockey")]
pd.DataFrame(
    [
        {"word 1": first, "word 2": second,
         "cosine similarity": word_vectors.similarity(first, second)}
        for first, second in word_pairs
    ]
)

,word 1,word 2,cosine similarity
0,Canada,hockey,0.276101
1,Japan,hockey,0.001963


### Analogies as vector arithmetic

Some relationships correspond approximately to directions in the embedding space. The classic example asks us to subtract the vector for *man* from *king*, then add the vector for *woman*:

$$
v(\text{king}) - v(\text{man}) + v(\text{woman}).
$$

We then find the word vector nearest to the result. The helper below expresses the question as “word 1 is to word 2 as word 3 is to what?”

In [6]:
def analogy(word1, word2, word3, model=word_vectors):
    result, similarity = model.most_similar(
        positive=[word2, word3], negative=[word1], topn=1
    )[0]
    return {"result": result, "cosine similarity": similarity}


analogy("man", "king", "woman")

{'result': 'queen', 'cosine similarity': 0.7118191719055176}

In [7]:
analogy("Montreal", "Canadiens", "Vancouver")

{'result': 'Canucks', 'cosine similarity': 0.8213266134262085}

These examples are striking, but they do not demonstrate human-like reasoning. They show that some regularities in the training corpus became approximately linear relationships among vectors. Other analogy questions produce unstable, irrelevant, or nonsensical results.

### Embeddings also encode bias

Learned associations are not always desirable. Before running the next cell, predict its result. What occupational relationship might the model infer?

In [8]:
analogy("man", "computer_programmer", "woman")

{'result': 'homemaker', 'cosine similarity': 0.5627118945121765}

The result, `homemaker`, reflects a gender stereotype in the news corpus and the model trained from it. It does **not** tell us that homemaking is the female equivalent of computer programming. The model has compressed patterns from its training data—including unequal representation and harmful social associations—into its geometry.

This example comes from an older embedding model and is unusually easy to expose with vector arithmetic. Some later models use data filtering or bias-mitigation methods, but we should not conclude that modern embeddings are bias-free. Bias can be subtle, depends on how a model is used, and must be evaluated in the context of the application.

### Exercise 17.1: Interpreting word embeddings

For each claim below, decide whether the preceding demonstrations provide sufficient evidence. Explain your reasoning.

1. Words with high cosine similarity are synonyms.
2. An analogy result reveals a true relationship between the concepts.
3. Removing one known gender analogy would make the embedding unbiased.
4. Nearest neighbours learned from Google News will necessarily be the most useful neighbours in another domain.

### From words to sentences and documents

Traditional word2vec gives each vocabulary item one vector. The word *bank* therefore has the same representation in *bank account* and *river bank*. It also does not directly give us one vector for a new sentence or document. Averaging its word vectors is possible, but doing so loses word order and often loses important context.

For information retrieval, we want to compare a user's query with sentences, passages, or documents. Modern **text-embedding models** map each complete piece of text to a dense vector. Texts expressing similar meanings tend to receive similar vectors even when they use different words.

These embeddings remain imperfect. They can miss specialized terminology, subtle negation, or important contextual distinctions, and they can reproduce biases in their training data. We should treat similarity as a useful model output, not as proof that two texts mean the same thing.

### Measuring similarity

To use embeddings for search, we need to compare vectors. A common measure is **cosine similarity**:

$$
\operatorname{cosine\_similarity}(a,b)
= \frac{a \cdot b}{\lVert a 
\lVert_2\lVert b 
\lVert_2}.
$$

Cosine similarity compares the directions of two vectors rather than their magnitudes. Larger values indicate greater similarity. For the embedding models used here, we rank documents from the largest cosine similarity to the smallest.

The value is meaningful only relative to the model and corpus. A score of 0.7 is not a 70% probability that a document is relevant.

## Information retrieval

**Information retrieval** (IR) is the task of finding useful items in a collection in response to a query. A search system usually returns a ranking rather than a single answer. In our example, the items are course-policy documents.

We will compare two approaches:

- **Sparse retrieval** represents queries and documents using terms, such as TF–IDF features.
- **Dense retrieval** represents queries and documents using embeddings from a pretrained model.

Neither is universally better. Sparse retrieval is especially effective for exact names, identifiers, and specialized terms. Dense retrieval can recognize paraphrases and conceptually related language. Many production search systems combine both.

### Sparse retrieval with TF–IDF

We fit the vectorizer on the document collection. At query time, we transform the new query using that fitted vocabulary and rank documents by cosine similarity.

In [9]:
tfidf = TfidfVectorizer(stop_words="english")
document_tfidf = tfidf.fit_transform(documents["text"])


def sparse_retrieve(query, k=3):
    query_tfidf = tfidf.transform([query])
    scores = cosine_similarity(query_tfidf, document_tfidf).ravel()
    top_indices = np.argsort(scores)[::-1][:k]
    results = documents.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results[["doc_id", "title", "score", "text"]]


sparse_retrieve("Where can I find lecture recordings?", k=3)

,doc_id,title,score,text
3,recordings,Lecture recordings,0.436197,Lecture recordings are posted on the course media page after class. Recordings are a supplement rather than a replac...
4,regrade,Regrade requests,0.000000,Submit a regrade request through the grading portal within seven days of receiving your mark. Explain the specific g...
2,collaboration,Collaboration,0.000000,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."


This query shares the words *lecture* and *recordings* with the relevant document, so lexical matching should work well. Now consider a paraphrased question.

In [10]:
sparse_retrieve("I was sick when my homework was due. What should I do?", k=3)

,doc_id,title,score,text
4,regrade,Regrade requests,0.0,Submit a regrade request through the grading portal within seven days of receiving your mark. Explain the specific g...
3,recordings,Lecture recordings,0.0,Lecture recordings are posted on the course media page after class. Recordings are a supplement rather than a replac...
2,collaboration,Collaboration,0.0,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."


Before looking at the result, predict which terms the vectorizer can match. The sparse retriever might still succeed, but it cannot directly infer that *sick* is related to *illness* or that *homework* is a kind of *assignment*.

### Dense retrieval with a pretrained embedding model

We next use a pretrained sentence-embedding model. It accepts a list of strings and returns one vector per string. Downloading the model the first time requires an internet connection; subsequent runs can use the locally cached copy.

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(
    documents["text"].tolist(), normalize_embeddings=True
)
document_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)

The model produces one dense vector for each document. Because we requested normalized vectors, their dot products are equal to their cosine similarities.

In [12]:
def dense_retrieve(query, k=3):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, document_embeddings).ravel()
    top_indices = np.argsort(scores)[::-1][:k]
    results = documents.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results[["doc_id", "title", "score", "text"]]


dense_retrieve("I was sick when my homework was due. What should I do?", k=3)

,doc_id,title,score,text
0,late-work,Late work,0.319222,Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency preven...
2,collaboration,Collaboration,0.318132,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."
1,office-hours,Office hours,0.174682,Teaching assistants hold office hours on Mondays and Wednesdays. Office hours are for conceptual questions and debug...


The result is a ranking, not a guarantee of relevance. Always inspect several queries from the intended application. In a larger system, document embeddings would normally be computed once and stored in a vector index. The index speeds up search but does not change the basic operation: find vectors close to the query vector.

### Evaluating retrieval

An appealing example is not enough to establish that a retriever works. We need evaluation queries with relevance judgments—records of which documents should count as relevant.

For a small illustration, suppose each query below has one relevant policy. **Recall@\(k\)** asks whether that document occurs anywhere in the first \(k\) results. **Reciprocal rank** is \(1/r\), where \(r\) is the rank of the first relevant result. It rewards putting relevant material near the top.

In [13]:
evaluation_queries = pd.DataFrame(
    [
        ("Where are class videos posted?", "recordings"),
        ("I need help debugging my code.", "office-hours"),
        ("Can my classmate send me their solution?", "collaboration"),
        ("How can I dispute a marking decision?", "regrade"),
        ("An emergency stopped me from submitting on time.", "late-work"),
    ],
    columns=["query", "relevant_doc_id"],
)


def evaluate_retriever(retriever, queries, k=3):
    records = []
    for row in queries.itertuples(index=False):
        retrieved_ids = retriever(row.query, k=k)["doc_id"].tolist()
        rank = (
            retrieved_ids.index(row.relevant_doc_id) + 1
            if row.relevant_doc_id in retrieved_ids
            else None
        )
        records.append(
            {
                "query": row.query,
                f"recall@{k}": int(rank is not None),
                "reciprocal_rank": 0 if rank is None else 1 / rank,
                "retrieved": retrieved_ids,
            }
        )
    return pd.DataFrame(records)


dense_evaluation = evaluate_retriever(dense_retrieve, evaluation_queries)
display(dense_evaluation)
print("Mean reciprocal rank:", dense_evaluation["reciprocal_rank"].mean())

,query,recall@3,reciprocal_rank,retrieved
0,Where are class videos posted?,1,1.000000,"[recordings, collaboration, late-work]"
1,I need help debugging my code.,1,0.333333,"[collaboration, recordings, office-hours]"
2,Can my classmate send me their solution?,1,1.000000,"[collaboration, late-work, recordings]"
3,How can I dispute a marking decision?,1,1.000000,"[regrade, late-work, collaboration]"
4,An emergency stopped me from submitting on time.,1,1.000000,"[late-work, office-hours, recordings]"


Mean reciprocal rank: 0.8666666666666666


These five queries are useful for understanding the calculation, but they are far too few to support a serious performance claim. Real evaluation data should represent the users, language, and difficult cases the system will encounter. Relevance can also be subjective, and some queries have several relevant documents.

### Exercise 17.2: Compare retrieval methods

Before running the retrievers, predict which method is likely to work better for each query. Then run both methods and explain their rankings.

1. `What is the deadline for a regrade request?`
2. `A family emergency made me miss the cutoff.`
3. `Can a TA finish the broken part of my program?`

Identify at least one query for which exact word overlap is valuable and one for which recognizing a paraphrase is valuable.

## Language models and LLMs

A **language model** learns patterns in sequences of text. Given some preceding text, a generative language model assigns probabilities to possible next tokens. A token may be a word, part of a word, punctuation, or another unit chosen by the model. Generation repeatedly selects a next token and appends it to the existing text.

A **large language model (LLM)** is a language model trained with a large number of parameters on a large collection of data. Many models are subsequently **instruction-tuned** so that they respond more usefully to requests written as instructions.

For this chapter, we can treat a language model through its interface:

```python
generated_text = language_model(prompt)
```

Understanding and using a RAG pipeline does not require knowing the internal neural-network architecture. We do, however, need to understand what information enters the prompt and what the generated answer can—and cannot—tell us.

### What does the model know?

The model's parameters contain patterns learned during training, not a dependable database of facts. A model may lack recent, private, or course-specific information. Even when relevant information appeared in its training data, it may produce an unsupported statement that sounds plausible. This behaviour is often called **hallucination**.

A model can also use text supplied in its **context**, including the prompt, conversation, and retrieved documents. The context has a finite size, so inserting an entire document collection is usually impractical. Extra irrelevant text can also distract the model and increase latency and cost.

This creates the motivation for RAG: retrieve a small amount of relevant evidence first, then ask the model to answer using that evidence.

## Retrieval-augmented generation

**Retrieval-augmented generation (RAG)** combines an information-retrieval system with a generative language model:

1. **Retrieve:** find text chunks relevant to the user's query.
2. **Augment:** place those chunks, the query, and instructions in a prompt.
3. **Generate:** ask a language model to answer from the supplied context.

RAG lets an application use material that was not part of the language model's training data. It can make answers easier to update and support with sources. It does not guarantee correctness: the retriever can miss necessary evidence, and the generator can ignore or misrepresent retrieved evidence.

### Step 1: Split documents into chunks

Long documents are commonly divided into smaller **chunks** before they are embedded. Small chunks can isolate a precise passage, but they may omit context needed to interpret it. Large chunks retain more context, but they may mix several topics and consume more of the model's context window. An overlap can preserve sentences near a boundary at the cost of storing repeated text.

Our policies are already short. The simple word-based function below makes the operation visible. Production systems often use token-aware or structure-aware splitting so that headings and paragraphs are not broken arbitrarily.

In [14]:
def split_into_chunks(text, chunk_size=50, overlap=10):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[start : start + chunk_size])
            for start in range(0, len(words), step)]


chunk_records = []
for document in documents.itertuples(index=False):
    for chunk_number, chunk_text in enumerate(split_into_chunks(document.text)):
        chunk_records.append(
            {
                "chunk_id": f"{document.doc_id}-{chunk_number}",
                "doc_id": document.doc_id,
                "title": document.title,
                "text": chunk_text,
            }
        )

chunks = pd.DataFrame(chunk_records)
chunks

,chunk_id,doc_id,title,text
0,late-work-0,late-work,Late work,Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency preven...
1,late-work-1,late-work,Late work,your message.
2,office-hours-0,office-hours,Office hours,Teaching assistants hold office hours on Mondays and Wednesdays. Office hours are for conceptual questions and debug...
3,collaboration-0,collaboration,Collaboration,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."
4,recordings-0,recordings,Lecture recordings,Lecture recordings are posted on the course media page after class. Recordings are a supplement rather than a replac...
5,regrade-0,regrade,Regrade requests,Submit a regrade request through the grading portal within seven days of receiving your mark. Explain the specific g...


### Step 2: Embed and store the chunks

We embed chunks rather than whole documents because these are the units the retriever will return. The DataFrame acts as our small document store, and the NumPy array acts as our vector store.

In [15]:
chunk_embeddings = embedding_model.encode(
    chunks["text"].tolist(), normalize_embeddings=True
)


def retrieve_chunks(query, k=3):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, chunk_embeddings).ravel()
    top_indices = np.argsort(scores)[::-1][:k]
    results = chunks.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results[["chunk_id", "doc_id", "title", "score", "text"]]


retrieve_chunks("I had an emergency and missed the assignment deadline.", k=3)

,chunk_id,doc_id,title,score,text
0,late-work-0,late-work,Late work,0.564383,Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency preven...
3,collaboration-0,collaboration,Collaboration,0.332089,"You may discuss assignment ideas with classmates, but the work you submit must be your own. Do not share code or wri..."
2,office-hours-0,office-hours,Office hours,0.286929,Teaching assistants hold office hours on Mondays and Wednesdays. Office hours are for conceptual questions and debug...


### Step 3: Construct an augmented prompt

The prompt tells the model how to use the retrieved text. In applications where unsupported answers are costly, it is useful to instruct the model to say when the context is insufficient. We also attach source identifiers so that the answer can point back to evidence.

In [16]:
def build_rag_prompt(query, retrieved_chunks):
    context_blocks = []
    for row in retrieved_chunks.itertuples(index=False):
        context_blocks.append(
            f"SOURCE: {row.chunk_id} ({row.title})\n{row.text}"
        )
    context = "\n\n".join(context_blocks)

    return f"""Answer the question using only the context below.
If the context does not contain enough information, say that you do not know.
Cite supporting source IDs in square brackets.

CONTEXT
{context}

QUESTION
{query}

ANSWER
"""


example_query = "What should I do if an emergency makes my assignment late?"
example_chunks = retrieve_chunks(example_query, k=1)
print(build_rag_prompt(example_query, example_chunks))

Answer the question using only the context below.
If the context does not contain enough information, say that you do not know.
Cite supporting source IDs in square brackets.

CONTEXT
SOURCE: late-work-0 (Late work)
Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency prevents you from submitting, contact the course coordinator before the deadline when possible. Do not include private medical details in your message.

QUESTION
What should I do if an emergency makes my assignment late?

ANSWER



Inspecting the prompt is an important debugging step. The language model cannot use evidence that the retriever failed to include. Notice also that the retrieved text is untrusted input. A document could contain instructions such as “ignore the user's question.” Robust applications must distinguish application instructions from document content and defend against this form of **prompt injection**.

### Step 4: Generate an answer

The code below uses a small instruction-following language model so that the example can run on an ordinary computer. It is not an LLM by contemporary standards, but it exposes the same input-output boundary used by a larger local model or a hosted LLM API. The retrieval code does not depend on which generator we choose.

The first run downloads the model. Model output can vary across library and model versions.

In [17]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

generator_name = "google/flan-t5-small"
generator_tokenizer = AutoTokenizer.from_pretrained(generator_name)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(generator_name)


def generate_text(prompt, max_new_tokens=100):
    model_inputs = generator_tokenizer(prompt, return_tensors="pt", truncation=True)
    output_ids = generator_model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return generator_tokenizer.decode(output_ids[0], skip_special_tokens=True)


def answer_with_rag(query, k=3):
    retrieved = retrieve_chunks(query, k=k)
    prompt = build_rag_prompt(query, retrieved)
    answer = generate_text(prompt)
    return {
        "answer": answer,
        "sources": retrieved[["chunk_id", "title", "score", "text"]],
        "prompt": prompt,
    }


rag_result = answer_with_rag(
    "What should I do if an emergency makes my assignment late?", k=1
)
print(rag_result["answer"])
display(rag_result["sources"])

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Contact the course coordinator before the deadline


,chunk_id,title,score,text
0,late-work-0,Late work,0.568489,Assignments may be submitted up to 48 hours after the deadline with a 10% penalty. If illness or an emergency preven...


Returning the retrieved sources alongside the answer is deliberate. A source list does not prove that the answer is supported; users or evaluation code must still check that the cited passages actually entail the claims being made.

### The complete pipeline

Our implementation contains two phases.

During **indexing**, we:

1. load documents;
2. split them into chunks;
3. embed the chunks; and
4. store the text, metadata, and vectors.

For each query, we:

1. embed the query;
2. retrieve the top-\(k\) chunks;
3. construct a prompt containing those chunks;
4. generate an answer; and
5. return the answer with its sources.

Indexing is normally done only when documents are added or changed. Query processing happens for every user request.

## Evaluating and improving a RAG pipeline

A final answer can fail for different reasons:

- **Retrieval failure:** the necessary evidence was not among the retrieved chunks.
- **Context failure:** a chunk contained relevant words but omitted information needed to interpret them.
- **Generation failure:** the evidence was present, but the model ignored, distorted, or contradicted it.
- **Knowledge-base failure:** the required information was absent, incorrect, or outdated in the source documents.

These failures call for different fixes. Changing the prompt will not recover a document that was never retrieved. Changing the embedding model will not correct an inaccurate source document. Evaluate intermediate outputs instead of treating the system as one black box.

### Important design choices

Several choices affect both quality and cost:

- **Embedding model:** It should suit the language and domain of the documents.
- **Chunking:** Chunk size, overlap, and document structure determine what can be retrieved together.
- **Retrieval method:** Sparse, dense, and hybrid methods have different strengths.
- **Number of chunks:** Increasing \(k\) may recover more evidence, but it also adds irrelevant context and consumes space.
- **Prompt:** The instructions should specify the task, permitted evidence, desired output, and behaviour when evidence is missing.
- **Generator:** Models differ in instruction-following ability, context size, latency, cost, and deployment constraints.

Changing any of these components should be followed by evaluation on representative queries.

### Risks and limitations

RAG provides access to external information; it does not make the overall system inherently trustworthy.

- Retrieved documents can contain private, copyrighted, biased, malicious, or outdated information.
- Embedding models may perform unevenly across languages and social groups.
- A generated answer can make claims that are absent from—or contradicted by—the retrieved evidence.
- Document text can attempt to manipulate the generator through prompt injection.
- Logging queries and retrieved passages may expose sensitive information.
- Model calls and large context windows introduce latency, monetary cost, and environmental cost.

The appropriate safeguards depend on the application. High-stakes systems may require access controls, document provenance, citations, abstention, human review, and monitoring after deployment.

### Exercise 17.3: Diagnose a failed answer

A student asks, “Can I submit ten days late if I accept a penalty?” The retriever returns only the *Late work* policy, but the generator answers, “Yes, with a 10% penalty.”

1. Does the retrieved passage support the answer?
2. Is the main failure in retrieval, generation, or the knowledge base?
3. What should a better answer say?

Explain your reasoning using the policy text rather than the model's confidence or writing style.

### Exercise 17.4: Extend the pipeline

Add two short documents to the knowledge base and write three realistic questions about them. At least one question should use different wording from its relevant document.

Then:

1. rebuild the chunk embeddings;
2. inspect the top-three chunks for each query;
3. record whether the relevant chunk was retrieved;
4. generate an answer only after checking retrieval; and
5. identify one change that improves the pipeline and one trade-off introduced by that change.

## Summary

Text must be represented numerically before a machine learning system can work with it. Word2vec embeddings illustrate how relationships among words can be encoded geometrically—and how unwanted social biases can be encoded at the same time. Sparse representations such as TF–IDF emphasize word overlap, while text embeddings can capture some semantic similarity between differently worded passages. Information-retrieval systems use these representations to rank documents or chunks for a query.

A generative language model produces text from a prompt, but its parameters are not a dependable or up-to-date knowledge base. RAG connects retrieval and generation: it retrieves relevant chunks, adds them to the prompt, and asks a language model to answer using that context.

The intermediate steps matter. A useful RAG workflow inspects and evaluates retrieval separately from generation, returns evidence with answers, and recognizes that retrieved context can still be incomplete, misleading, or unsafe.